# Skill Broad Buckets by Company AI Strategy

Classifies companies into **Augmentation**, **Automation**, **Both**, or **Other** (not in classification file),
then generates Skill Broad Bucket time-series charts for each category.

Skill taxonomy from: `ksao_full_mapping_with_buckets.csv` (6 broad buckets)
Company classification from: Raisch & Krakowski (2021) via `500.xlsx`

In [ ]:
# =============================================================================
# CONFIGURATION & IMPORTS
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter, defaultdict
import json
import ast
import gc
import glob
from typing import List, Dict, Any
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

INPUT_FOLDER = '../data/extracted_skills_production'
SKILLS_COLUMN = 'skills'
DATE_COLUMN = 'post_date'
RCID_COLUMN = 'rcid'
CLASSIFICATION_FILE = '500.xlsx'
BUCKET_MAPPING_FILE = '../skillner/data/ksao_full_mapping_with_buckets.csv'

# --- Helper function ---
def parse_skills_column(value) -> List[str]:
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return [str(s) for s in value if s is not None]
    if isinstance(value, str):
        if value in ('[]', '', 'nan', 'None'):
            return []
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            pass
    return []

# --- Load Broad Bucket Lookup ---
bucket_df = pd.read_csv(BUCKET_MAPPING_FILE)
# Build skill_name (lowercase) -> broad_bucket mapping
BUCKET_LOOKUP = dict(zip(bucket_df['skill_name'].str.lower().str.strip(),
                         bucket_df['broad_bucket']))
print(f"Loaded bucket mapping with {len(BUCKET_LOOKUP):,} skills")
print(f"Broad buckets: {sorted(bucket_df['broad_bucket'].dropna().unique())}")

# --- Find parquet files ---
parquet_files = sorted(glob.glob(f'{INPUT_FOLDER}/*.parquet'))
print(f"Found {len(parquet_files)} parquet files")

In [ ]:
# =============================================================================
# LOAD & PROCESS COMPANY CLASSIFICATIONS FROM 500.xlsx
# =============================================================================

import openpyxl

wb = openpyxl.load_workbook(CLASSIFICATION_FILE, read_only=True, data_only=True)
ws = wb['Sheet1']

headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
print("Columns:", headers)

rcid_col = headers.index('rcid')
class_col = headers.index('Raisch & Krakowski (2021) Classification')
name_col = headers.index('Company name')

VALID_CLASSES = {'Augmentation', 'Automation', 'Both'}

company_rows = []
for row in ws.iter_rows(min_row=2, values_only=True):
    rcid_val = row[rcid_col]
    classification = row[class_col]
    company_name = row[name_col]
    if rcid_val is None or str(rcid_val).strip() in ('#N/A', '', 'N/A', 'nan'):
        continue
    if classification not in VALID_CLASSES:
        continue
    try:
        rcid_int = int(float(str(rcid_val)))
    except (ValueError, TypeError):
        continue
    company_rows.append((rcid_int, company_name, classification))

wb.close()

rcid_classes = defaultdict(set)
rcid_names = {}
for rcid_int, company_name, classification in company_rows:
    rcid_classes[rcid_int].add(classification)
    rcid_names[rcid_int] = company_name

rcid_to_category = {}
for rcid_int, classes in rcid_classes.items():
    if 'Both' in classes:
        rcid_to_category[rcid_int] = 'Both'
    elif 'Augmentation' in classes and 'Automation' in classes:
        rcid_to_category[rcid_int] = 'Both'
    elif 'Automation' in classes:
        rcid_to_category[rcid_int] = 'Automation'
    else:
        rcid_to_category[rcid_int] = 'Augmentation'

category_counts = Counter(rcid_to_category.values())
print(f"\nCompany classifications (by unique RCID):")
print(f"  Total classified companies: {len(rcid_to_category)}")
for cat in ['Augmentation', 'Automation', 'Both']:
    print(f"  {cat}: {category_counts.get(cat, 0)} companies")

rcid_to_category_str = {str(k): v for k, v in rcid_to_category.items()}
print(f"\nReady to match against parquet files.")

In [ ]:
# =============================================================================
# STREAM PARQUET FILES: Compute Quarterly Broad Bucket Averages by Category
# =============================================================================

CATEGORIES = ['Augmentation', 'Automation', 'Both', 'Other']
BUCKET_NAMES = sorted(bucket_df['broad_bucket'].dropna().unique().tolist())
print(f"Broad buckets ({len(BUCKET_NAMES)}): {BUCKET_NAMES}")

def make_empty_quarter_dict():
    d = {b: 0 for b in BUCKET_NAMES}
    d['__jd_count__'] = 0
    return d

cat_quarterly = {cat: defaultdict(make_empty_quarter_dict) for cat in CATEGORIES}

# Per-RCID quarterly accumulator (classified companies only)
rcid_quarterly = defaultdict(lambda: defaultdict(make_empty_quarter_dict))

# Per-category skill counts for summary tables
cat_bucket_skill_counts = {cat: {b: Counter() for b in BUCKET_NAMES + ['Unknown']} for cat in CATEGORIES}
cat_matched = {cat: 0 for cat in CATEGORIES}
cat_unmatched = {cat: 0 for cat in CATEGORIES}

print("Scanning parquet files for quarterly Broad Bucket by company category...")

for i, filepath in enumerate(parquet_files):
    try:
        df_part = pd.read_parquet(filepath, columns=[RCID_COLUMN, DATE_COLUMN, SKILLS_COLUMN])
    except Exception as e:
        print(f"  Skipping {filepath}: {e}")
        continue
    
    df_part[DATE_COLUMN] = pd.to_datetime(df_part[DATE_COLUMN], errors='coerce')
    df_part['quarter'] = df_part[DATE_COLUMN].dt.to_period('Q')
    
    for _, row in df_part.iterrows():
        q = row['quarter']
        if pd.isna(q):
            continue
        q_str = str(q)
        
        rcid_val = row[RCID_COLUMN]
        rcid_str = str(int(float(rcid_val))) if pd.notna(rcid_val) else ''
        category = rcid_to_category_str.get(rcid_str, 'Other')
        
        cat_quarterly[category][q_str]['__jd_count__'] += 1
        
        is_classified = rcid_str in rcid_to_category_str
        if is_classified:
            rcid_quarterly[rcid_str][q_str]['__jd_count__'] += 1
        
        skills_list = parse_skills_column(row.get(SKILLS_COLUMN))
        for skill in skills_list:
            skill_lower = skill.lower().strip()
            if skill_lower in BUCKET_LOOKUP:
                bucket = BUCKET_LOOKUP[skill_lower]
                if bucket in cat_quarterly[category][q_str]:
                    cat_quarterly[category][q_str][bucket] += 1
                if is_classified and bucket in rcid_quarterly[rcid_str][q_str]:
                    rcid_quarterly[rcid_str][q_str][bucket] += 1
                cat_bucket_skill_counts[category][bucket][skill] += 1
                cat_matched[category] += 1
            else:
                cat_bucket_skill_counts[category]['Unknown'][skill] += 1
                cat_unmatched[category] += 1
    
    del df_part
    gc.collect()
    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(parquet_files)}] processed")

print(f"\nDone. Building DataFrames...")

# Build column names from bucket names
bucket_col_names = [f'bucket_{b.replace(" ", "_").replace("&", "and").lower()}' for b in BUCKET_NAMES]

quarterly_dfs = {}
for cat in CATEGORIES:
    quarters_sorted = sorted(cat_quarterly[cat].keys())
    rows_list = []
    for q in quarters_sorted:
        jd_count = cat_quarterly[cat][q]['__jd_count__']
        if jd_count == 0:
            continue
        row_data = {'year_quarter': q}
        for bname, col_name in zip(BUCKET_NAMES, bucket_col_names):
            row_data[col_name] = cat_quarterly[cat][q][bname] / jd_count
        rows_list.append(row_data)
    quarterly_dfs[cat] = pd.DataFrame(rows_list)
    n_jds = sum(cat_quarterly[cat][q]['__jd_count__'] for q in cat_quarterly[cat])
    print(f"  {cat}: {len(quarterly_dfs[cat])} quarters, {n_jds:,} total JDs")

print(f"\nPer-RCID quarterly data collected for {len(rcid_quarterly)} classified companies.")
print("Ready for summary and visualization.")

In [ ]:
# =============================================================================
# SKILLS SUMMARY BY COMPANY CATEGORY x BROAD BUCKET
# =============================================================================

for cat in CATEGORIES:
    matched = cat_matched[cat]
    unmatched = cat_unmatched[cat]
    total = matched + unmatched
    
    cat_label = f'{cat.upper()} COMPANIES' if cat != 'Other' else 'OTHER COMPANIES (not classified)'
    
    print("=" * 70)
    print(f"SKILLS BY {cat_label} BY BROAD BUCKET")
    print("=" * 70)
    print(f"\nTotal skill mentions: {total:,}")
    if total > 0:
        print(f"Matched to bucket: {matched:,} ({matched/total*100:.1f}%)")
        print(f"Unmatched: {unmatched:,} ({unmatched/total*100:.1f}%)")
    
    print(f"\n{'Broad Bucket':<35} {'Mentions':<12} {'% of Total':<12} {'Unique Skills'}")
    print("-" * 75)
    
    for b in BUCKET_NAMES + ['Unknown']:
        count = sum(cat_bucket_skill_counts[cat][b].values())
        unique = len(cat_bucket_skill_counts[cat][b])
        pct = (count / total) * 100 if total > 0 else 0
        print(f"{b:<35} {count:<12,} {pct:<12.1f} {unique:,}")
    
    print()

In [ ]:
# =============================================================================
# QUARTERLY AVERAGE TABLE BY COMPANY CATEGORY + CSV EXPORT
# =============================================================================

for cat in CATEGORIES:
    cat_label = f'{cat}' if cat != 'Other' else 'Other (not classified)'
    print(f"Average skills per JD by {cat_label} companies by Broad Bucket (quarterly):")
    qdf = quarterly_dfs.get(cat)
    if qdf is not None and len(qdf) > 0:
        print(qdf.to_string(index=False))
    else:
        print("  No data available.")
    print()

# --- CSV Export ---

# 1. Skills summary CSV
summary_rows = []
for cat in CATEGORIES:
    total = cat_matched[cat] + cat_unmatched[cat]
    for b in BUCKET_NAMES + ['Unknown']:
        count = sum(cat_bucket_skill_counts[cat][b].values())
        unique = len(cat_bucket_skill_counts[cat][b])
        pct = (count / total) * 100 if total > 0 else 0
        summary_rows.append({
            'category': cat,
            'broad_bucket': b,
            'mentions': count,
            'pct_of_total': round(pct, 1),
            'unique_skills': unique
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('skills_by_bucket_per_category.csv', index=False)
print(f"Saved: skills_by_bucket_per_category.csv ({len(summary_df)} rows)")

# 2. Quarterly CSV
quarterly_all = []
for cat in CATEGORIES:
    qdf = quarterly_dfs.get(cat)
    if qdf is not None and len(qdf) > 0:
        qdf_copy = qdf.copy()
        qdf_copy.insert(0, 'category', cat)
        quarterly_all.append(qdf_copy)
if quarterly_all:
    quarterly_combined = pd.concat(quarterly_all, ignore_index=True)
    quarterly_combined.to_csv('quarterly_bucket_per_category.csv', index=False)
    print(f"Saved: quarterly_bucket_per_category.csv ({len(quarterly_combined)} rows)")
else:
    print("No quarterly data to export.")

In [ ]:
# =============================================================================
# PER-RCID CSV EXPORT: Quarterly Average Skills for Each Classified Company
# =============================================================================
import os, re

output_dir_avg = 'rcid_quarterly_buckets_avg'
os.makedirs(output_dir_avg, exist_ok=True)

exported_count = 0

for rcid_str, quarter_data in rcid_quarterly.items():
    quarters_sorted = sorted(quarter_data.keys())
    rows_list = []
    for q in quarters_sorted:
        jd_count = quarter_data[q]['__jd_count__']
        if jd_count == 0:
            continue
        row_data = {'year_quarter': q, 'jd_count': jd_count}
        for bname, col_name in zip(BUCKET_NAMES, bucket_col_names):
            row_data[col_name] = quarter_data[q][bname] / jd_count
        rows_list.append(row_data)
    
    if not rows_list:
        continue
    
    rcid_int = int(rcid_str)
    company_name = rcid_names.get(rcid_int, 'unknown')
    category = rcid_to_category.get(rcid_int, 'Unknown')
    
    safe_name = re.sub(r'[^\w\s-]', '', str(company_name)).strip().replace(' ', '_')
    filename = f'{rcid_str}_{safe_name}.csv'
    
    df_rcid = pd.DataFrame(rows_list)
    df_rcid.insert(0, 'rcid', rcid_str)
    df_rcid.insert(1, 'company_name', company_name)
    df_rcid.insert(2, 'category', category)
    df_rcid.to_csv(os.path.join(output_dir_avg, filename), index=False)
    exported_count += 1

print(f"Exported {exported_count} per-RCID quarterly average CSV files to '{output_dir_avg}/'")
print(f"Columns: rcid, company_name, category, year_quarter, jd_count, {', '.join(bucket_col_names)}")

In [ ]:
# =============================================================================
# VISUALIZATION: 4 Individual Charts (one per category) - from 2019Q1 onward
# =============================================================================

# Color palette for 6 buckets
bucket_mapping = {}
bucket_colors = ['#e74c3c', '#3498db', '#9b59b6', '#27ae60', '#f39c12', '#1abc9c']
for col_name, bname, color in zip(bucket_col_names, BUCKET_NAMES, bucket_colors):
    bucket_mapping[col_name] = (bname, color)

def plot_bucket_over_time(quarterly_ai, title):
    """Generate stacked area + line chart for one category."""
    if quarterly_ai is None or len(quarterly_ai) == 0:
        print(f"No data for: {title}")
        return
    
    # Filter to >= 2019Q1
    quarterly_ai = quarterly_ai[quarterly_ai['year_quarter'] >= '2019Q1'].reset_index(drop=True)
    if len(quarterly_ai) == 0:
        print(f"No data after 2019Q1 for: {title}")
        return
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    x_labels = [str(q) for q in quarterly_ai['year_quarter']]
    
    # 1. Stacked area chart
    ax1 = axes[0]
    bottom = np.zeros(len(quarterly_ai))
    for col, (label, color) in bucket_mapping.items():
        if col in quarterly_ai.columns:
            values = quarterly_ai[col].values
            ax1.fill_between(range(len(x_labels)), bottom, bottom + values,
                             label=label, color=color, alpha=0.7)
            bottom = bottom + values
    ax1.set_xticks(range(0, len(x_labels), max(1, len(x_labels)//15)))
    ax1.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), max(1, len(x_labels)//15))],
                        rotation=45, ha='right')
    ax1.set_ylabel('Average Skills per JD')
    ax1.set_title(f'{title} \u2014 Composition Over Time', fontsize=12)
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    ax1.grid(axis='y', alpha=0.3)
    
    # 2. Line chart with markers
    ax2 = axes[1]
    for col, (label, color) in bucket_mapping.items():
        if col in quarterly_ai.columns:
            ax2.plot(range(len(x_labels)), quarterly_ai[col].values,
                     label=label, color=color, linewidth=2, marker='o', markersize=3)
    ax2.set_xticks(range(0, len(x_labels), max(1, len(x_labels)//15)))
    ax2.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), max(1, len(x_labels)//15))],
                        rotation=45, ha='right')
    ax2.set_xlabel('Quarter')
    ax2.set_ylabel('Average Skills per JD')
    ax2.set_title(f'{title} \u2014 Trends Over Time', fontsize=12)
    ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print trend summary
    print(f"\n{title} \u2014 Change from first to last quarter:")
    print("-" * 65)
    for col, (label, _) in bucket_mapping.items():
        if col in quarterly_ai.columns and len(quarterly_ai) > 1:
            first_val = quarterly_ai[col].iloc[1] if len(quarterly_ai) > 1 else quarterly_ai[col].iloc[0]
            last_val = quarterly_ai[col].iloc[-1]
            change = ((last_val - first_val) / first_val * 100) if first_val > 0 else 0
            trend = '\u2191' if change > 0 else '\u2193' if change < 0 else '\u2192'
            print(f"  {label:<35}: {first_val:.2f} \u2192 {last_val:.2f} ({trend} {abs(change):.1f}%)")
    print()

# Count companies per category
cat_company_counts = Counter(rcid_to_category.values())

# Generate 4 charts
for cat in CATEGORIES:
    n_companies = cat_company_counts.get(cat, 0)
    if cat == 'Other':
        label = f'Other Companies (not in classification)'
    else:
        label = f'{cat} Companies (N={n_companies})'
    plot_bucket_over_time(quarterly_dfs.get(cat), f'Skill Broad Buckets \u2014 {label}')

In [ ]:
# =============================================================================
# COMBINED 2x2 COMPARISON CHART - from 2019Q1 onward, shared y-axis per row
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

cat_company_counts = Counter(rcid_to_category.values())

# First pass: plot data and collect y-max per row
row_ymax = [0.0, 0.0]

for idx, cat in enumerate(CATEGORIES):
    row_idx = idx // 2
    col_idx = idx % 2
    ax = axes[row_idx][col_idx]
    qdf = quarterly_dfs.get(cat)
    
    if qdf is None or len(qdf) == 0:
        ax.text(0.5, 0.5, f'No data for {cat}', ha='center', va='center', fontsize=14)
        ax.set_title(cat)
        continue
    
    # Filter to >= 2019Q1
    qdf = qdf[qdf['year_quarter'] >= '2019Q1'].reset_index(drop=True)
    if len(qdf) == 0:
        ax.text(0.5, 0.5, f'No data after 2019Q1 for {cat}', ha='center', va='center', fontsize=14)
        ax.set_title(cat)
        continue
    
    x_labels = [str(q) for q in qdf['year_quarter']]
    
    for col, (label, color) in bucket_mapping.items():
        if col in qdf.columns:
            vals = qdf[col].values
            ax.plot(range(len(x_labels)), vals,
                    label=label, color=color, linewidth=2, marker='o', markersize=2)
            row_ymax[row_idx] = max(row_ymax[row_idx], vals.max())
    
    tick_step = max(1, len(x_labels) // 10)
    ax.set_xticks(range(0, len(x_labels), tick_step))
    ax.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), tick_step)],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Avg Skills per JD', fontsize=9)
    ax.grid(alpha=0.3)
    
    n_companies = cat_company_counts.get(cat, 0)
    if cat == 'Other':
        ax.set_title(f'Other (not classified)', fontsize=11)
    else:
        ax.set_title(f'{cat} (N={n_companies})', fontsize=11)

# Second pass: set shared y-axis limits per row
for row_idx in range(2):
    y_top = row_ymax[row_idx] * 1.05
    if y_top > 0:
        for col_idx in range(2):
            axes[row_idx][col_idx].set_ylim(0, y_top)

# Shared legend
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle('Skill Broad Buckets Over Time \u2014 By Company AI Strategy (2019Q1+)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# RAW COUNTS: Build Quarterly DataFrames (total JDs + total skill mentions)
# =============================================================================

raw_quarterly_dfs = {}
for cat in CATEGORIES:
    quarters_sorted = sorted(cat_quarterly[cat].keys())
    rows_list = []
    for q in quarters_sorted:
        jd_count = cat_quarterly[cat][q]['__jd_count__']
        if jd_count == 0:
            continue
        row_data = {'year_quarter': q, 'jd_count': jd_count}
        for bname, col_name in zip(BUCKET_NAMES, bucket_col_names):
            row_data[col_name] = cat_quarterly[cat][q][bname]
        rows_list.append(row_data)
    raw_quarterly_dfs[cat] = pd.DataFrame(rows_list)
    total_jds = sum(r['jd_count'] for r in rows_list)
    print(f"  {cat}: {len(raw_quarterly_dfs[cat])} quarters, {total_jds:,} total JDs")

print("\nRaw count DataFrames ready.")

In [ ]:
# =============================================================================
# RAW COUNTS: Quarterly Tables + CSV Export
# =============================================================================

for cat in CATEGORIES:
    cat_label = f'{cat}' if cat != 'Other' else 'Other (not classified)'
    print(f"Raw skill counts per quarter — {cat_label} companies:")
    rdf = raw_quarterly_dfs.get(cat)
    if rdf is not None and len(rdf) > 0:
        print(rdf.to_string(index=False))
    else:
        print("  No data available.")
    print()

# CSV Export — raw counts
raw_all = []
for cat in CATEGORIES:
    rdf = raw_quarterly_dfs.get(cat)
    if rdf is not None and len(rdf) > 0:
        rdf_copy = rdf.copy()
        rdf_copy.insert(0, 'category', cat)
        raw_all.append(rdf_copy)
if raw_all:
    raw_combined = pd.concat(raw_all, ignore_index=True)
    raw_combined.to_csv('quarterly_bucket_raw_counts.csv', index=False)
    print(f"Saved: quarterly_bucket_raw_counts.csv ({len(raw_combined)} rows)")
else:
    print("No raw count data to export.")

In [ ]:
# =============================================================================
# PER-RCID CSV EXPORT: Total Bucket Counts (non-quarterly, all-time totals)
# =============================================================================
import os, re

output_dir = 'rcid_quarterly_buckets'
os.makedirs(output_dir, exist_ok=True)

exported_count = 0

for rcid_str, quarter_data in rcid_quarterly.items():
    # Sum across all quarters for this RCID
    total_jd_count = 0
    bucket_totals = {b: 0 for b in BUCKET_NAMES}
    for q_str, q_data in quarter_data.items():
        total_jd_count += q_data['__jd_count__']
        for bname in BUCKET_NAMES:
            bucket_totals[bname] += q_data[bname]
    
    if total_jd_count == 0:
        continue
    
    rcid_int = int(rcid_str)
    company_name = rcid_names.get(rcid_int, 'unknown')
    category = rcid_to_category.get(rcid_int, 'Unknown')
    
    safe_name = re.sub(r'[^\\w\\s-]', '', str(company_name)).strip().replace(' ', '_')
    filename = f'{rcid_str}_{safe_name}.csv'
    
    row_data = {
        'rcid': rcid_str,
        'company_name': company_name,
        'category': category,
        'jd_count': total_jd_count
    }
    for bname, col_name in zip(BUCKET_NAMES, bucket_col_names):
        row_data[col_name] = bucket_totals[bname]
    
    df_rcid = pd.DataFrame([row_data])
    df_rcid.to_csv(os.path.join(output_dir, filename), index=False)
    exported_count += 1

print(f"Exported {exported_count} per-RCID total bucket CSV files to '{output_dir}/'")
print(f"Each file: 1 row with rcid, company_name, category, jd_count, + 6 bucket raw counts")

In [ ]:
# =============================================================================
# RAW COUNTS VISUALIZATION: 4 Individual Charts (one per category) - 2019Q1+
# =============================================================================

def plot_bucket_raw_counts(raw_df, title):
    """Generate stacked area + line chart for raw skill counts."""
    if raw_df is None or len(raw_df) == 0:
        print(f"No data for: {title}")
        return
    
    # Filter to >= 2019Q1
    raw_df = raw_df[raw_df['year_quarter'] >= '2019Q1'].reset_index(drop=True)
    if len(raw_df) == 0:
        print(f"No data after 2019Q1 for: {title}")
        return
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 14))
    x_labels = [str(q) for q in raw_df['year_quarter']]
    tick_idx = range(0, len(x_labels), max(1, len(x_labels)//15))
    tick_labels = [x_labels[i] for i in tick_idx]
    
    # 1. JD count bar chart
    ax0 = axes[0]
    ax0.bar(range(len(x_labels)), raw_df['jd_count'].values, color='#34495e', alpha=0.7)
    ax0.set_xticks(list(tick_idx))
    ax0.set_xticklabels(tick_labels, rotation=45, ha='right')
    ax0.set_ylabel('Number of JDs')
    ax0.set_title(f'{title} \u2014 JD Count per Quarter', fontsize=12)
    ax0.grid(axis='y', alpha=0.3)
    
    # 2. Stacked area chart (raw counts)
    ax1 = axes[1]
    bottom = np.zeros(len(raw_df))
    for col, (label, color) in bucket_mapping.items():
        if col in raw_df.columns:
            values = raw_df[col].values
            ax1.fill_between(range(len(x_labels)), bottom, bottom + values,
                             label=label, color=color, alpha=0.7)
            bottom = bottom + values
    ax1.set_xticks(list(tick_idx))
    ax1.set_xticklabels(tick_labels, rotation=45, ha='right')
    ax1.set_ylabel('Total Skill Mentions')
    ax1.set_title(f'{title} \u2014 Composition (Raw Counts)', fontsize=12)
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    ax1.grid(axis='y', alpha=0.3)
    
    # 3. Line chart (raw counts)
    ax2 = axes[2]
    for col, (label, color) in bucket_mapping.items():
        if col in raw_df.columns:
            ax2.plot(range(len(x_labels)), raw_df[col].values,
                     label=label, color=color, linewidth=2, marker='o', markersize=3)
    ax2.set_xticks(list(tick_idx))
    ax2.set_xticklabels(tick_labels, rotation=45, ha='right')
    ax2.set_xlabel('Quarter')
    ax2.set_ylabel('Total Skill Mentions')
    ax2.set_title(f'{title} \u2014 Trends (Raw Counts)', fontsize=12)
    ax2.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

cat_company_counts = Counter(rcid_to_category.values())

for cat in CATEGORIES:
    n_companies = cat_company_counts.get(cat, 0)
    if cat == 'Other':
        label = f'Other Companies (not in classification)'
    else:
        label = f'{cat} Companies (N={n_companies})'
    plot_bucket_raw_counts(raw_quarterly_dfs.get(cat), f'Skill Broad Buckets \u2014 {label}')

In [ ]:
# =============================================================================
# RAW COUNTS: 2x2 COMPARISON CHART - 2019Q1+, shared y-axis per row
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
cat_company_counts = Counter(rcid_to_category.values())

row_ymax = [0.0, 0.0]

for idx, cat in enumerate(CATEGORIES):
    row_idx = idx // 2
    col_idx = idx % 2
    ax = axes[row_idx][col_idx]
    rdf = raw_quarterly_dfs.get(cat)
    
    if rdf is None or len(rdf) == 0:
        ax.text(0.5, 0.5, f'No data for {cat}', ha='center', va='center', fontsize=14)
        ax.set_title(cat)
        continue
    
    rdf = rdf[rdf['year_quarter'] >= '2019Q1'].reset_index(drop=True)
    if len(rdf) == 0:
        ax.text(0.5, 0.5, f'No data after 2019Q1 for {cat}', ha='center', va='center', fontsize=14)
        ax.set_title(cat)
        continue
    
    x_labels = [str(q) for q in rdf['year_quarter']]
    
    for col, (label, color) in bucket_mapping.items():
        if col in rdf.columns:
            vals = rdf[col].values
            ax.plot(range(len(x_labels)), vals,
                    label=label, color=color, linewidth=2, marker='o', markersize=2)
            row_ymax[row_idx] = max(row_ymax[row_idx], vals.max())
    
    tick_step = max(1, len(x_labels) // 10)
    ax.set_xticks(range(0, len(x_labels), tick_step))
    ax.set_xticklabels([x_labels[i] for i in range(0, len(x_labels), tick_step)],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Total Skill Mentions', fontsize=9)
    ax.grid(alpha=0.3)
    
    n_companies = cat_company_counts.get(cat, 0)
    if cat == 'Other':
        ax.set_title(f'Other (not classified)', fontsize=11)
    else:
        ax.set_title(f'{cat} (N={n_companies})', fontsize=11)

for row_idx in range(2):
    y_top = row_ymax[row_idx] * 1.05
    if y_top > 0:
        for col_idx in range(2):
            axes[row_idx][col_idx].set_ylim(0, y_top)

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle('Skill Broad Buckets Over Time (Raw Counts) \u2014 By Company AI Strategy (2019Q1+)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()